# Step 2c · Robustness checks

Step 2b's main result: counties with 72+ hours of outage in a week had far fewer visits than counties without a substantial outage, and wind and rain controls took up only a small part of the gap. This step tests whether the result survives four objections. To have enough observations, 72–120 h and 120+ h are merged into one band (**72+ h**), and every check looks at the coefficient of that band.

| Check | Objection | Approach |
| --- | --- | --- |
| 1 Hazard effects that vary by week | Step 2b's cumulative wind and rain variables assume the storm's effect is as large in week 6 as in week 0; storm effects not taken up in the first two weeks may have been attributed to outages | Each band of cumulative wind and rain is interacted with "weeks since the storm", so the storm's effect can fade week by week |
| 2 County × event fixed effects | The counties with the longest outages are also the most damaged | Each county gets its own intercept in each event, so only changes within a county from week to week are used: after the wind stops the damage no longer changes, but outage duration changes as repairs go on |
| 3 Inference | County-weeks at 72+ h come from a few dozen counties, mostly in Ida; county-clustered standard errors may be too small | Wild cluster bootstrap p-values; re-estimate without Ida and without Laura |
| 4 Inland counties only | Storm-surge flooding is not among the controls | Using the Census coastline, drop counties within 25 km of the Atlantic or Gulf coast (10, 50 and 100 km reported too) |

Each check is run with two sets of wind and rain controls: Step 2b's controls (**S3**) and the week-varying controls of check 1 (**HxW**). The decision rule looks at the HxW results, because they control the storm itself more strictly.

**Structure**: A data → B the decision rule, fixed in advance → C estimation functions → D check 1 and the event study → E check 2 → F check 3 → G check 4 → H summary, two figures, checklist. No Earth Engine is needed; two shapefiles (about 15 MB) are downloaded from the Census Bureau. The bootstrap repeats 999 times per specification; everything takes only a minute or two.

In [ ]:
# 1. Environment, folders, chart style
import os, sys, glob, zipfile, subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm
import requests
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz", "step2a_hazard_weekly.csv"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run Steps 1b and 2a first.")

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
HOLDOUT = ["uri2021"]                          # as in Step 2b, kept out of the main sample
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]       # all bands below are the same as in Step 2b
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
GUST_EDGES = [-np.inf, 17.5, 25, 33, np.inf]
RAIN_EDGES = [-np.inf, 25, 75, 150, np.inf]
COLD_EDGES = [-np.inf, 5, 25, np.inf]
RAINCUM_EDGES = [-np.inf, 50, 150, 300, np.inf]
CUM_FROM = -1
ES_REF = -2
TEST = "out4=h072p"                            # the coefficient every check looks at: 72+ h vs under 6 h
B_BOOT = 999                                   # bootstrap replications
SEED = 2026
COAST_KM = [10, 25, 50, 100]                   # drop counties this close to the coast
COAST_RULE = 25                                # the distance used in the decision rule
Z = norm.ppf(0.975)                            # 95% intervals from the normal distribution, as in Step 2b

## A. Data

The same merge, bands and sample as Step 2b, plus a few variables:

- `out4`: outages in four bands, with 72+ h merged into one
- `pw`: weeks since the storm, with week −2 and earlier merged into "pre" and week 4 and later into "4+"
- `gw`, `rw`: the cumulative gust band and the cumulative rain band combined with `pw` (for check 1). Below tropical-storm gusts, or under 50 mm of cumulative rain, counts as the reference in every week
- `ce`: county × event (for check 2)

In [ ]:
# 3. Merge, bands, sample (as in Step 2b); new variables
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
for c in ["t0", "day0", "week0"]:
    EVENTS[c] = pd.to_datetime(EVENTS[c])
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])

D = (W.merge(L[["event", "fips", "name", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left"))
D["out_bin"]     = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["out4"]        = D["out_bin"].replace({"h072": "h072p", "h120": "h072p"})
D["gust_bin"]    = pd.cut(D["gust_mean_ms"], GUST_EDGES, labels=["g0", "g1", "g2", "g3"]).astype(str)
D["gustcum_bin"] = pd.cut(D["gust_cum"], GUST_EDGES, labels=["k0", "k1", "k2", "k3"]).astype(str)
D["rain_bin"]    = pd.cut(D["rain_mm"], RAIN_EDGES, labels=["r0", "r1", "r2", "r3"]).astype(str)
D["raincum_bin"] = pd.cut(D["rain_cum"], RAINCUM_EDGES, labels=["s0", "s1", "s2", "s3"]).astype(str)
D["cold_bin"]    = pd.cut(D["cold_ppt_mm"], COLD_EDGES, labels=["c0", "c1", "c2"]).astype(str)
D["ew"]  = D["event"] + "_" + D["rel_week"].astype(str)                       # event × week
D["ce"]  = D["event"] + "_" + D["fips"]                                       # county × event
D["pw"]  = ["pre" if r <= -2 else ("4+" if r >= 4 else str(r)) for r in D["rel_week"]]
D["gw"]  = np.where(D["gustcum_bin"] == "k0", "base", D["gustcum_bin"] + "|" + D["pw"])
D["rw"]  = np.where(D["raincum_bin"] == "s0", "base", D["raincum_bin"] + "|" + D["pw"])
D["ewk"] = D["ew"] + "_" + D["gustcum_bin"]                                   # event × week × cumulative gust band
D["es"]  = ["ref" if g not in ("ge72", "24to72") or r == ES_REF else f"{g}_{'m' if r < 0 else 'p'}{abs(r)}"
            for g, r in zip(D["group"], D["rel_week"])]                        # event study: group × week
NEEDED = ["out_week_hours", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D[NEEDED].notna().all(axis=1)
MAIN = D[D["keep"] & ~D["event"].isin(HOLDOUT)].copy()
print(f"Main sample: {len(MAIN):,} county-weeks; 72+ h: {int((MAIN['out4'] == 'h072p').sum())} county-weeks in "
      f"{MAIN.loc[MAIN['out4'] == 'h072p', 'fips'].nunique()} counties")
pd.crosstab(MAIN["event"], MAIN["out4"]).reindex(columns=["h000", "h006", "h024", "h072p"], fill_value=0)

## B. The decision rule, fixed in advance

This rule is written down before looking at any of the checks. "Significantly negative" means a coefficient below 0 with a 95% interval that excludes 0. All of the following use the HxW controls.

1. **Check 1**: with all counties, the 72+ h coefficient is significantly negative
2. **Check 2**: with county × event fixed effects added, it is still significantly negative
3. **Check 3**: the wild cluster bootstrap p-value is below 0.05; without Ida and without Laura, it is still significantly negative in each case
4. **Check 4**: without counties within 25 km of the coast, it is still significantly negative

**If all four hold**: the conclusion stands, and can be written as "when outages approach or exceed three days, local activity falls by more than measurable wind and rain can explain, and this holds in four robustness checks".

**If any one fails**: the conclusion is downgraded to "more than measurable wind and rain can explain, but sensitive to [the failed check]", and the write-up says which check failed and why it might have.

Two notes:

- Check 2 estimates "how many fewer visits the same county had in a week with 72+ h of outage", a narrower quantity than the Step 2b coefficient, so a smaller number is expected; it passes as long as it is still significantly negative
- Without coastal counties, the county-weeks at 72+ h drop sharply and the interval widens; a failure there may simply reflect too few observations, so the number of county-weeks left is reported alongside

## C. Estimation functions

Step 2b wrote fixed effects as dummy variables in statsmodels; with over three thousand county × event effects that is too slow. Here the fixed effects are first "subtracted" (each group minus its mean, alternating between two sets of fixed effects until convergence), and least squares is run afterwards. The result is identical to writing out the dummies; with a single set of fixed effects, coefficients and standard errors also match Step 2b's statsmodels exactly.

**Wild cluster bootstrap** (Webb weights, null imposed): first a model with "72+ h coefficient = 0" is estimated, giving fitted values and residuals; each replication draws a random weight for each county, multiplies it into that county's residuals to create a new outcome, re-estimates and computes the t-statistic. The p-value is the share of the 999 replications more extreme than the real t-statistic. It does not rely on "many counties", so it is more reliable when only a few dozen counties are treated.

In [ ]:
# 4. Fixed-effects least squares, clustered standard errors, wild cluster bootstrap
OUT_X = [("out4", "h000")]
NOW_X = [("gust_bin", "g0"), ("rain_bin", "r0"), ("cold_bin", "c0")]
CUM_X = [("gustcum_bin", "k0"), ("raincum_bin", "s0")]
HXW_X = [("gw", "base"), ("rw", "base")]
CONTROLS = {"S3": NOW_X + CUM_X, "HxW": NOW_X + HXW_X}
WEBB = np.array([-np.sqrt(1.5), -1.0, -np.sqrt(0.5), np.sqrt(0.5), 1.0, np.sqrt(1.5)])

def dummies(d, specs):
    # banded variables → 0/1 columns (reference band dropped), named "variable=band"
    parts = [pd.DataFrame(index=d.index)]
    for col, ref in specs:
        dm = pd.get_dummies(d[col].astype(str), prefix=col, prefix_sep="=", dtype=float)
        parts.append(dm.drop(columns=[f"{col}={ref}"], errors="ignore"))
    return pd.concat(parts, axis=1)

def demean(M, codes, tol=1e-10, max_iter=5000):
    # remove fixed effects: subtract group means for each set in turn until convergence (one pass is enough for a single set)
    M = np.array(M, dtype=float)
    groups = [(c, sp.csr_matrix((np.ones(len(c)), (c, np.arange(len(c))))), np.bincount(c)) for c in codes]
    scale = max(1.0, float(np.abs(M).max()))
    for _ in range(max_iter):
        change = 0.0
        for c, S, n in groups:
            means = (S @ M) / n[:, None]
            M -= means[c]
            change = max(change, float(np.abs(means).max()))
        if len(groups) <= 1 or change < tol * scale:
            break
    return M

def drop_singletons(d, fe):
    # observations alone in a fixed-effect group add nothing; drop them repeatedly until none are left
    while True:
        lone = np.zeros(len(d), bool)
        for f in fe:
            lone |= (d.groupby(f)[f].transform("size") < 2).to_numpy()
        if not lone.any():
            return d
        d = d[~lone]

def fe_ols(data, y, xspec, fe=("ew",), test=TEST, boot=0, seed=SEED, keep_draws=False):
    # fixed-effects least squares; county-clustered standard errors (CR1, as statsmodels); optional wild cluster bootstrap
    d = drop_singletons(data.dropna(subset=[y]), list(fe))
    X = dummies(d, xspec)
    X = X.loc[:, X.sum() > 0]
    codes = [pd.factorize(d[f])[0] for f in fe]
    Zm = demean(np.column_stack([d[y].to_numpy(float), X.to_numpy()]), codes)
    yv, Xm = Zm[:, 0], Zm[:, 1:]
    live = np.linalg.norm(Xm, axis=0) > 1e-7 * np.sqrt(len(yv))               # drop columns fully absorbed by the fixed effects
    names, Xm = list(X.columns[live]), Xm[:, live]
    N = len(yv)
    XtX_inv = np.linalg.pinv(Xm.T @ Xm)
    A = XtX_inv @ Xm.T
    b = A @ yv
    u = yv - Xm @ b
    g = pd.factorize(d["fips"])[0]
    G = int(g.max()) + 1
    C = sp.csr_matrix((np.ones(N), (g, np.arange(N))), shape=(G, N))
    nested = [f for f in fe if d.groupby(f)["fips"].nunique().max() == 1]   # fixed effects nested within counties are not counted in the degrees of freedom (as in Stata's reghdfe)
    k_fe = sum(d[f].nunique() for f in fe if f not in nested) - max(0, len(fe) - 1)
    K = int(np.linalg.matrix_rank(Xm)) + k_fe
    corr = G / (G - 1) * (N - 1) / (N - K)
    meat = C @ (Xm * u[:, None])
    se = np.sqrt(np.clip(np.diag(corr * XtX_inv @ (meat.T @ meat) @ XtX_inv), 0, None))
    table = pd.DataFrame({"coef": b, "se": se, "lo": b - Z * se, "hi": b + Z * se}, index=names)
    col, lev = test.split("=")
    treated = d[col].astype(str) == lev
    out = {"table": table, "n": N, "clusters": G, "used": d, "p_boot": np.nan,
           "n_test": int(treated.sum()), "counties_test": int(d.loc[treated, "fips"].nunique())}
    if boot and test in names:
        k = names.index(test)
        Xr = np.delete(Xm, k, axis=1)
        fit_r = Xr @ np.linalg.lstsq(Xr, yv, rcond=None)[0]                   # the model with the null (coefficient = 0) imposed
        u_r = yv - fit_r
        a_k = A[k]
        t_obs = b[k] / se[k]
        rng = np.random.default_rng(seed)
        t_star, draws = [], []
        for start in range(0, boot, 100):
            v = rng.choice(WEBB, size=(G, min(100, boot - start)))           # one weight per county
            Ys = fit_r[:, None] + u_r[:, None] * v[g]
            Bs = A @ Ys
            Us = Ys - Xm @ Bs
            s = C @ (a_k[:, None] * Us)
            t_star.append(Bs[k] / np.sqrt(corr * (np.asarray(s) ** 2).sum(axis=0)))
            if keep_draws:
                draws.append(v)
        t_star = np.concatenate(t_star)
        out["p_boot"] = float(np.mean(np.abs(t_star) >= abs(t_obs)))
        if keep_draws:
            out.update(draws=np.hstack(draws), t_star=t_star, t_obs=t_obs)
    return out

ROWS = []
def run(check, sample, data, ctrl, fe=("ew",), boot=B_BOOT):
    # run one specification and store the 72+ h coefficient in the summary
    r = fe_ols(data, "visits_in_chg", OUT_X + CONTROLS[ctrl], fe, boot=boot)
    t = r["table"].reindex([TEST]).iloc[0]
    row = {"check": check, "sample": sample, "controls": ctrl, "fe": " + ".join(fe), "coef": t["coef"], "se": t["se"],
           "lo": t["lo"], "hi": t["hi"], "p_boot": r["p_boot"], "n_obs": r["n"], "n_72plus": r["n_test"], "counties_72plus": r["counties_test"]}
    ROWS.append(row)
    print(f"  {sample:<36} {ctrl:<4} {row['coef']:+7.1f} [{row['lo']:+6.1f}, {row['hi']:+6.1f}]  bootstrap p = {row['p_boot']:.3f}  "
          f"n = {row['n_72plus']} county-weeks / {row['counties_72plus']} counties")
    return row

## D. Check 1: hazard effects that vary by week, and the event study

Two approaches:

- **HxW**: each band of cumulative wind and rain is interacted with `pw` (weeks since the storm)
- **Event × week × cumulative-gust-band fixed effects**: stricter, comparing only counties in the same event, week and cumulative gust band; not in the decision rule, for reference

Then Step 2b's event study is redone with three sets of controls (none, S3, HxW). If the positive values of the 72+ h group in weeks 3–6 in Step 2b really came from how the cumulative variables were set up, they should return to around 0 with HxW.

In [ ]:
# 5. Check 1
for ctrl in CONTROLS:
    run("1", "all counties", MAIN, ctrl)
for ctrl in CONTROLS:
    run("1", "event × week × gust band FE", MAIN, ctrl, fe=("ewk",))
two = fe_ols(MAIN, "visits_in_chg", [("out_bin", "h000")] + CONTROLS["HxW"])["table"]
print("\nWith HxW controls, the two bands separately:")
tidy(two.loc[[c for c in two.index if c.startswith("out_bin=")]], 1)

In [ ]:
# 6. Event study: three sets of controls
es_rows = []
for ctrl, xs in [("none", []), ("S3", CONTROLS["S3"]), ("HxW", CONTROLS["HxW"])]:
    t = fe_ols(MAIN, "visits_in_chg", [("es", "ref")] + xs, test="es=ge72_p1")["table"]
    t = t[t.index.str.startswith("es=")].copy()
    t["group"] = [i[3:].split("_")[0] for i in t.index]
    t["rel_week"] = [(-1 if s[0] == "m" else 1) * int(s[1:]) for s in (i.split("_")[1] for i in t.index)]
    t["controls"] = ctrl
    es_rows.append(t)
    for grp in ["ge72", "24to72"]:
        es_rows.append(pd.DataFrame([{"group": grp, "rel_week": ES_REF, "coef": 0.0, "se": 0.0, "lo": 0.0, "hi": 0.0, "controls": ctrl}]))
es = pd.concat(es_rows, ignore_index=True).sort_values(["controls", "group", "rel_week"]).reset_index(drop=True)
es.to_csv(f"{OUT}/step2c_event_study.csv", index=False)
late = es[(es["group"] == "ge72") & es["rel_week"].between(3, 6)].groupby("controls")["coef"].mean()
print("72+ h group, mean coefficient in weeks 3–6:", late.round(1).to_dict())
tidy(es[es["group"] == "ge72"].pivot(index="rel_week", columns="controls", values="coef"), 1)

In [ ]:
# 7. Figure 1: the event study under three sets of controls
STY = {"none": (C_GREY, "--", "No wind and rain controls"),
       "S3": (C_LIGHT, "-", "Step 2b controls (same effect every week)"),
       "HxW": (C_DARK, "-", "Controls that vary by week (check 1)")}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)
for ax, (grp, title) in zip(axes, [("ge72", "Counties with outages of 72 h or more"), ("24to72", "Counties with outages of 24–72 h")]):
    for ctrl, (color, ls, label) in STY.items():
        s = es[(es["group"] == grp) & (es["controls"] == ctrl)].sort_values("rel_week")
        off = {"none": -0.12, "S3": 0.0, "HxW": 0.12}[ctrl]
        if ctrl == "HxW":
            ax.errorbar(s["rel_week"] + off, s["coef"], yerr=[s["coef"] - s["lo"], s["hi"] - s["coef"]], color=color, lw=2,
                        marker="o", ms=4, elinewidth=1.2, capsize=0, label=label)
        else:
            ax.plot(s["rel_week"] + off, s["coef"], color=color, lw=1.6, ls=ls, marker="o" if ctrl == "S3" else None, ms=3.5, label=label)
    ax.axhline(0, color=FAINT, lw=1)
    ax.axvline(0, color=FAINT, lw=1)
    ax.set_xticks(range(-5, 7))
    ax.set_title(title)
    ax.set_xlabel(f"Week (week 0 contains day 0; week {ES_REF} is the reference)")
axes[0].set_ylabel("Visits to places in the county,\nlog points vs other counties")
handles, labs = axes[0].get_legend_handles_labels()
order = [labs.index(STY[k][2]) for k in ["HxW", "S3", "none"]]
axes[0].legend([handles[i] for i in order], [labs[i] for i in order], fontsize=8.5, loc="lower left")
fig.suptitle("Event study under three sets of wind and rain controls, 7 events (95% intervals for the week-varying controls)",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step2c_fig1_event_study.png"); plt.show()

## E. Check 2: county × event fixed effects

Each county gets its own intercept in each event, so visits are compared only with other weeks of the same county: for example, Lafourche had 146 hours of outage in week 1 after Ida, 69 hours in week 3 and 22 hours in week 4, and the differences between those weeks are what is used. Once the wind stops, damage to homes and roads no longer changes, while power supply does, so this coefficient is closer to the effect of the outage itself. The cost is that part of the information in the weeks before the storm is absorbed by the intercept, so precision falls.

In [ ]:
# 8. Check 2
for ctrl in CONTROLS:
    run("2", "county × event FE", MAIN, ctrl, fe=("ce", "ew"))

## F. Check 3: inference and the influence of single events

Every specification above already has a bootstrap p-value. Here Ida and Laura are each left out in turn: Ida accounts for about 40% of the county-weeks at 72+ h, and nearly all of those at 120+ h.

In [ ]:
# 9. Check 3: without Ida, without Laura
for key, label in [("ida2021", "without Ida"), ("laura2020", "without Laura")]:
    for ctrl in CONTROLS:
        run("3", label, MAIN[MAIN["event"] != key], ctrl)

## G. Check 4: inland counties only

The 2020 coastline and county boundaries are downloaded from the Census Bureau (the 2020 county file still uses the old Connecticut county codes, consistent with the data so far). Only the Atlantic and Gulf coastlines are kept; the Great Lakes do not count. Each county's shortest distance to the coastline is computed in an equal-area projection (EPSG:5070); a county whose boundary touches the coastline has distance 0.

First, all counties with 72+ h of outage in any week are listed with their distance to the coast, showing which counties are dropped. Note: counties on Lake Pontchartrain (such as St. John the Baptist, flooded by the lake during Ida) may be more than 25 km from the Gulf coastline; check the distances in the table to see in which setting they are dropped.

In [ ]:
# 10. Download the coastline and county boundaries; distance from each county to the coast
try:
    import geopandas as gpd
except ImportError:                     # Colab usually has geopandas; install it if not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)
    import geopandas as gpd

GEO = f"{OUT}/step2c_geo"
os.makedirs(GEO, exist_ok=True)
SHAPES = {"coast": "https://www2.census.gov/geo/tiger/TIGER2020/COASTLINE/tl_2020_us_coastline.zip",
          "county": "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"}

def shapefile(kind):
    # download a Census Bureau shapefile (once), unzip it and read it
    path = f"{GEO}/{kind}.zip"
    if not os.path.exists(path) or os.path.getsize(path) < 10_000:
        r = requests.get(SHAPES[kind], timeout=600, headers={"User-Agent": "Mozilla/5.0 (research script)"})
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    folder = f"{GEO}/{kind}"
    with zipfile.ZipFile(path) as z:
        z.extractall(folder)
    return gpd.read_file(glob.glob(f"{folder}/*.shp")[0])

try:
    coast = shapefile("coast")
    print("Names of the coastline features:", coast["NAME"].value_counts().to_dict())
    sea = coast[coast["NAME"].astype(str).str.contains("Atlantic|Gulf", case=False)].to_crs(5070)
    shapes = shapefile("county")[["GEOID", "geometry"]]
    shapes = shapes[shapes["GEOID"].isin(set(D["fips"]))].to_crs(5070)
    near = gpd.sjoin_nearest(shapes, sea[["geometry"]], distance_col="dist_m")
    COAST = (near.groupby("GEOID")["dist_m"].min() / 1000).rename("coast_km")
    COAST.rename_axis("fips").to_csv(f"{OUT}/step2c_coast_distance.csv")
    missing = sorted(set(MAIN["fips"]) - set(COAST.index))
    print(f"Distances for {len(COAST):,} counties; counties in the main sample without a boundary: {len(missing)}", missing[:10])
except Exception as err:
    COAST = None
    print("Downloading the coastline or county boundaries failed; check 4 will show as pending.:", err)

if COAST is not None:
    MAIN["coast_km"] = MAIN["fips"].map(COAST)
    hit = (MAIN[MAIN["out4"] == "h072p"].groupby(["event", "fips", "name"])
           .agg(weeks_72plus=("rel_week", "size"), max_hours=("out_week_hours", "max"), lowest_visits=("visits_in_chg", "min"))
           .reset_index())
    hit["coast_km"] = hit["fips"].map(COAST)
    for km in COAST_KM:
        hit[f"kept_{km}km"] = hit["coast_km"] > km
    display(tidy(hit.sort_values("coast_km"), 0))

In [ ]:
# 11. Check 4
if COAST is not None:
    for km in COAST_KM:
        for ctrl in CONTROLS:
            run("4", f"inland, > {km} km from the coast", MAIN[MAIN["coast_km"] > km], ctrl)

## H. Summary

**Figure 2** shows the 72+ h coefficient in every specification: dark for the HxW controls (the set the decision rule uses), light for the S3 controls. Each row gives the number of county-weeks at 72+ h used and the bootstrap p-value under HxW. Rows marked * enter the decision rule.

In [ ]:
# 12. Summary table and verdict
R = pd.DataFrame(ROWS)
R.to_csv(f"{OUT}/step2c_robustness.csv", index=False)
RULE_ROWS = ["all counties", "county × event FE", "without Ida", "without Laura", f"inland, > {COAST_RULE} km from the coast"]

def row(sample, ctrl="HxW"):
    s = R[(R["sample"] == sample) & (R["controls"] == ctrl)]
    return s.iloc[0] if len(s) else None

def neg_sig(r):
    return None if r is None or pd.isna(r["coef"]) else bool(r["coef"] < 0 and r["hi"] < 0)

full = row("all counties")
parts = {
    "1": neg_sig(full),
    "2": neg_sig(row("county × event FE")),
    "3": bool(full["p_boot"] < 0.05 and neg_sig(row("without Ida")) and neg_sig(row("without Laura"))),
    "4": neg_sig(row(f"inland, > {COAST_RULE} km from the coast")),
}
failed = [k for k, v in parts.items() if v is False]
pending = [k for k, v in parts.items() if v is None]
print("Decision rule:", parts)
if pending:
    print("Conclusion: pending (no result for check " + ", ".join(pending) + ")")
elif not failed:
    print("The conclusion stands: when outages approach or exceed three days, local activity falls by more than measurable wind and rain can explain, and this holds in four robustness checks")
else:
    print("The conclusion is downgraded: more than measurable wind and rain can explain, but sensitive to check " + ", ".join(failed) + " (see the checklist)")
show = R.assign(cell=R.apply(lambda r: f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}]  p={r['p_boot']:.3f}  n={r['n_72plus']}", axis=1))
show.pivot(index="sample", columns="controls", values="cell").reindex([s for s in dict.fromkeys(R["sample"])])

In [ ]:
# 13. Figure 2: the 72+ h coefficient in every specification
ORDER = [s for s in dict.fromkeys(R["sample"])]
fig, ax = plt.subplots(figsize=(9.5, 0.6 * len(ORDER) + 1.6))
ys = np.arange(len(ORDER))[::-1]
labels = []
for y0, smp in zip(ys, ORDER):
    for ctrl, color, off, name in [("S3", C_LIGHT, 0.15, "Step 2b controls (S3)"), ("HxW", C_DARK, -0.15, "Week-varying controls (HxW)")]:
        r = row(smp, ctrl)
        if r is None or pd.isna(r["coef"]):
            continue
        ax.errorbar(r["coef"], y0 + off, xerr=[[r["coef"] - r["lo"]], [r["hi"] - r["coef"]]], fmt="o", color=color, ms=5.5,
                    elinewidth=1.6, capsize=0, label=name if y0 == ys[0] else None)
    h = row(smp)
    star = " *" if smp in RULE_ROWS else ""
    labels.append(f"{smp}{star}\n" + (f"n = {h['n_72plus']}, bootstrap p = {h['p_boot']:.3f}" if h is not None else ""))
ax.set_yticks(ys)
ax.set_yticklabels(labels, fontsize=8.5)
ax.axvline(0, color=FAINT, lw=1)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Visits to places in the county, log points: 72+ h of outage in the week vs under 6 h (95% interval)")
ax.legend(fontsize=8.5, loc="lower left")
ax.set_title("Robustness of the 72+ h coefficient, 7 events")
fig.tight_layout()
fig.text(0.01, -0.01, "* used in the decision rule (week-varying controls). n: county-weeks at 72+ h in the sample.", ha="left", va="top", color=MUTED, fontsize=8.5)
fig.savefig(f"{OUT}/step2c_fig2_robustness.png"); plt.show()

In [ ]:
# 14. Step 2c checklist
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

def cell(r):
    return "—" if r is None else f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}], n = {r['n_72plus']}"

inland = row(f"inland, > {COAST_RULE} km from the coast")
checks = [
    ("Check 1: HxW controls, all counties", cell(full), "significantly negative", parts["1"]),
    ("Check 2: county × event fixed effects", cell(row("county × event FE")), "significantly negative", parts["2"]),
    ("Check 3a: wild cluster bootstrap p-value", round(float(full["p_boot"]), 3), "< 0.05", bool(full["p_boot"] < 0.05)),
    ("Check 3b: without Ida", cell(row("without Ida")), "significantly negative", neg_sig(row("without Ida"))),
    ("Check 3c: without Laura", cell(row("without Laura")), "significantly negative", neg_sig(row("without Laura"))),
    (f"Check 4: counties more than {COAST_RULE} km from the coast", cell(inland), "significantly negative", parts["4"]),
    ("Reference: mean coefficient of the 72+ h group in weeks 3–6 (HxW)",
     f"{late.get('HxW', np.nan):+.1f} (S3: {late.get('S3', np.nan):+.1f})", "|·| ≤ 1.5", bool(abs(late.get("HxW", np.nan)) <= 1.5)),
    ("Reference: county-weeks at 72+ h in the inland sample", None if inland is None else int(inland["n_72plus"]), "≥ 15",
     None if inland is None else bool(inland["n_72plus"] >= 15)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step2c_checklist.csv", index=False)
check_tab